"# Member 3: Image Caption Generation\n",
    "## Baseline vs RAG (Group 11 - RAG Reranking)\n",
    "\n",
    "**Author:** Member 3  \n",
    "**Project:** Comparative study of RAG techniques for image captioning (MS COCO)  \n",
    "**Date:** November 2026\n",
    "\n",
    "---\n",
    "\n",
    "### Pipeline Overview\n",
    "1. Load BLIP-2 (Vision-Language Model)\n",
    "2. Load Member 2's FAISS index + Reranker\n",
    "3. Generate Baseline captions (no RAG)\n",
    "4. Generate RAG captions (with Top-5 retrieved examples)\n",
    "5. Save results to CSV for Member 4 (Evaluation)"

In [ ]:
"!pip install -q transformers accelerate sentence-transformers faiss-cpu pillow pandas tqdm"

In [ ]:
"import os\n",
    "import json\n",
    "import subprocess\n",
    "import urllib.request\n",
    "import torch\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import faiss\n",
    "\n",
    "from PIL import Image\n",
    "from pathlib import Path\n",
    "from tqdm import tqdm\n",
    "from transformers import (\n",
    "    Blip2Processor,\n",
    "    Blip2ForConditionalGeneration,\n",
    "    CLIPProcessor,\n",
    "    CLIPModel\n",
    ")\n",
    "from sentence_transformers import CrossEncoder\n",
    "\n",
    "print(\"✅ All libraries imported!\")"

In [ ]:
"REPO_URL = \"https://github.com/Rachaaabn/ImageCaptioning-RagReranking.git\"\n",
    "REPO_DIR = Path(\"/content/repo\")\n",
    "\n",
    "if not REPO_DIR.exists():\n",
    "    print(\"Cloning repository...\")\n",
    "    subprocess.run([\"git\", \"clone\", REPO_URL, str(REPO_DIR)], check=True)\n",
    "else:\n",
    "    print(\"Pulling latest changes...\")\n",
    "    subprocess.run([\"git\", \"-C\", str(REPO_DIR), \"pull\"], check=True)\n",
    "\n",
    "print(f\"✅ Repo ready at {REPO_DIR}\")"

In [ ]:
"print(\"Loading BLIP-2 (this may take a few minutes)...\")\n",
    "model_name = \"Salesforce/blip2-flan-t5-xl\"\n",
    "processor = Blip2Processor.from_pretrained(model_name)\n",
    "blip_model = Blip2ForConditionalGeneration.from_pretrained(\n",
    "    model_name,\n",
    "    torch_dtype=torch.float16,\n",
    "    device_map=\"auto\"\n",
    ")\n",
    "blip_model.eval()\n",
    "print(\"✅ BLIP-2 loaded!\")"

In [ ]:
"print(\"Loading CLIP (for retrieval)...\")\n",
    "clip_model = CLIPModel.from_pretrained(\"openai/clip-vit-base-patch32\")\n",
    "clip_processor = CLIPProcessor.from_pretrained(\"openai/clip-vit-base-patch32\")\n",
    "print(\"✅ CLIP loaded!\")"

In [ ]:
"# Load FAISS index\n",
    "index = faiss.read_index(str(REPO_DIR / \"outputs\" / \"faiss_index.bin\"))\n",
    "print(f\"✅ FAISS index loaded ({index.ntotal} vectors)\")\n",
    "\n",
    "# Load valid IDs\n",
    "with open(REPO_DIR / \"outputs\" / \"valid_kb_ids.json\", \"r\") as f:\n",
    "    valid_kb_ids = json.load(f)\n",
    "\n",
    "# Load captions for lookup\n",
    "with open(REPO_DIR / \"outputs\" / \"captions.json\", \"r\") as f:\n",
    "    all_captions = json.load(f)\n",
    "\n",
    "caption_lookup = {}\n",
    "for item in all_captions:\n",
    "    img_id = item[\"image_id\"]\n",
    "    if img_id not in caption_lookup:\n",
    "        caption_lookup[img_id] = []\n",
    "    caption_lookup[img_id].append(item[\"caption_original\"])\n",
    "\n",
    "print(f\"✅ Loaded {len(caption_lookup)} image captions for lookup\")"

In [ ]:
 "print(\"Loading Cross-Encoder Reranker...\")\n",
    "reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')\n",
    "print(\"✅ Reranker loaded!\")"

In [ ]:
"def get_rag_examples(test_image_path, top_k=5):\n",
    "    \"\"\"Retrieve Top-5 similar captions using FAISS + Reranker.\"\"\"\n",
    "    image = Image.open(test_image_path).convert(\"RGB\")\n",
    "    inputs = clip_processor(images=image, return_tensors=\"pt\")\n",
    "\n",
    "    with torch.no_grad():\n",
    "        features = clip_model.get_image_features(**inputs).pooler_output\n",
    "        query_vec = features.numpy().flatten()\n",
    "        query_vec = query_vec / np.linalg.norm(query_vec)\n",
    "        query_vec = np.array([query_vec]).astype(\"float32\")\n",
    "\n",
    "    # FAISS: Top 20 candidates\n",
    "    distances, indices = index.search(query_vec, 20)\n",
    "\n",
    "    candidates = []\n",
    "    for idx in indices[0]:\n",
    "        kb_id = valid_kb_ids[idx]\n",
    "        if kb_id in caption_lookup:\n",
    "            candidates.append(caption_lookup[kb_id][0])\n",
    "\n",
    "    # Reranker: keep Top 5\n",
    "    query_text = \"Describe this image\"\n",
    "    pairs = [[query_text, c] for c in candidates]\n",
    "    scores = reranker.predict(pairs)\n",
    "    ranked = sorted(zip(scores, candidates), reverse=True)\n",
    "    return [text for score, text in ranked[:top_k]]\n",
    "\n",
    "\n",
    "def generate_baseline_caption(image_path):\n",
    "    \"\"\"Generate caption without RAG.\"\"\"\n",
    "    image = Image.open(image_path).convert(\"RGB\")\n",
    "    inputs = processor(images=image, return_tensors=\"pt\").to(blip_model.device)\n",
    "    with torch.no_grad():\n",
    "        generated_ids = blip_model.generate(**inputs, max_new_tokens=50)\n",
    "    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]\n",
    "\n",
    "\n",
    "def generate_rag_caption(image_path):\n",
    "    \"\"\"Generate caption using Top-5 RAG examples.\"\"\"\n",
    "    examples = get_rag_examples(image_path, top_k=5)\n",
    "    examples_text = \"\\n\".join([f\"- {ex}\" for ex in examples])\n",
    "    prompt = (\n",
    "        f\"Here are some examples of similar images:\\n{examples_text}\\n\\n\"\n",
    "        \"Now, describe the new image based on the visual details, \"\n",
    "        \"using the style of the examples above.\"\n",
    "    )\n",
    "\n",
    "    image = Image.open(image_path).convert(\"RGB\")\n",
    "    inputs = processor(images=image, text=prompt, return_tensors=\"pt\").to(blip_model.device)\n",
    "    with torch.no_grad():\n",
    "        generated_ids = blip_model.generate(**inputs, max_new_tokens=50)\n",
    "    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]\n",
    "\n",
    "\n",
    "print(\"✅ All functions defined!\")"

In [ ]:
"# Load test IDs\n",
    "test_ids = json.load(open(REPO_DIR / \"outputs\" / \"splits.json\"))[\"test_ids\"]\n",
    "os.makedirs(\"/content/test_images\", exist_ok=True)\n",
    "\n",
    "# Download first test image\n",
    "first_id = test_ids[0]\n",
    "img_str = str(first_id).zfill(12)\n",
    "filepath = f\"/content/test_images/{img_str}.jpg\"\n",
    "\n",
    "if not os.path.exists(filepath):\n",
    "    urllib.request.urlretrieve(f\"http://images.cocodataset.org/val2017/{img_str}.jpg\", filepath)\n",
    "\n",
    "print(f\"Image: {filepath}\")\n",
    "print(f\"Baseline: {generate_baseline_caption(filepath)}\")\n",
    "print(f\"RAG:      {generate_rag_caption(filepath)}\")"

In [ ]:
    "results = []\n",
    "\n",
    "for test_id in tqdm(test_ids, desc=\"Processing test images\"):\n",
    "    img_str = str(test_id).zfill(12)\n",
    "    filepath = f\"/content/test_images/{img_str}.jpg\"\n",
    "\n",
    "    if not os.path.exists(filepath):\n",
    "        try:\n",
    "            urllib.request.urlretrieve(\n",
    "                f\"http://images.cocodataset.org/val2017/{img_str}.jpg\",\n",
    "                filepath\n",
    "            )\n",
    "        except Exception as e:\n",
    "            print(f\"Failed: {img_str} ({e})\")\n",
    "            continue\n",
    "\n",
    "    try:\n",
    "        baseline = generate_baseline_caption(filepath)\n",
    "    except Exception as e:\n",
    "        baseline = f\"ERROR: {e}\"\n",
    "\n",
    "    try:\n",
    "        rag = generate_rag_caption(filepath)\n",
    "    except Exception as e:\n",
    "        rag = f\"ERROR: {e}\"\n",
    "\n",
    "    results.append({\n",
    "        \"image_id\": test_id,\n",
    "        \"file_name\": f\"{img_str}.jpg\",\n",
    "        \"baseline_caption\": baseline,\n",
    "        \"rag_caption\": rag\n",
    "    })\n",
    "\n",
    "df = pd.DataFrame(results)\n",
    "df.to_csv(\"/content/rag_captions_comparison.csv\", index=False)\n",
    "print(f\"\\n✅ Saved {len(df)} captions!\")\n",
    "display(df.head())"


In [ ]:
"from google.colab import files\n",
    "files.download(\"/content/rag_captions_comparison.csv\")"